In [1]:
# Cell 1: Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import joblib
import os

print("✅ Step 1: All required libraries imported successfully!")

✅ Step 1: All required libraries imported successfully!


In [2]:
# Cell 2: Generate the CSV dataset and load it
np.random.seed(42)
n_samples = 5000

# Generating realistic dummy data
df_gen = pd.DataFrame({
    'zone_id': np.random.randint(1, 6, n_samples),
    'density': np.random.randint(50, 800, n_samples), 
    'speed': np.random.uniform(0.1, 2.5, n_samples)   
})

# Feature Engineering for labeling
df_gen['bottleneck_score'] = df_gen['density'] / (df_gen['speed'] + 0.1)

def assign_risk(row):
    if row['bottleneck_score'] > 500 and row['density'] > 400:
        return 2  # CRITICAL
    elif row['bottleneck_score'] > 250:
        return 1  # WARNING
    else:
        return 0  # NORMAL

df_gen['risk_label'] = df_gen.apply(assign_risk, axis=1)

# Drop the bottleneck_score so the models learn to predict solely using density and speed
df_gen.drop(columns=['bottleneck_score'], inplace=True)

# Save to CSV
csv_filename = 'crowd_data.csv'
df_gen.to_csv(csv_filename, index=False)
print(f"✅ Step 2a: Dataset generated and saved locally as '{csv_filename}'.")

# Load the dataset from the CSV
df = pd.read_csv(csv_filename)
print("✅ Step 2b: Dataset loaded successfully from CSV.")
print(df.head())

✅ Step 2a: Dataset generated and saved locally as 'crowd_data.csv'.
✅ Step 2b: Dataset loaded successfully from CSV.
   zone_id  density     speed  risk_label
0        4      712  1.659673           1
1        5      449  0.159274           2
2        3      120  2.044622           0
3        5      229  1.387788           0
4        5      173  0.767721           0


In [3]:
# Cell 3: Train-Test Split and Feature Scaling

# Features (X) and Target (y)
X = df[['density', 'speed']]
y = df['risk_label']

# Split the data (80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("✅ Step 3: Data successfully split into training and testing sets, and features have been scaled.")

✅ Step 3: Data successfully split into training and testing sets, and features have been scaled.


In [4]:
# Cell 4: Train 4 Models, Compare Accuracies, and Select the Best

models = {
    "Logistic Regression": LogisticRegression(random_state=42),
    "Support Vector Machine (SVM)": SVC(probability=True, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, random_state=42)
}

best_model_name = ""
best_model_accuracy = 0
best_model = None

print("🏟️ Starting Model Competition...\n" + "-"*40)

for name, model in models.items():
    # Train the model
    model.fit(X_train_scaled, y_train)
    
    # Make predictions
    y_pred = model.predict(X_test_scaled)
    
    # Evaluate accuracy
    acc = accuracy_score(y_test, y_pred)
    print(f"🔹 {name} Accuracy: {acc * 100:.2f}%")
    
    # Auto-select the best model
    if acc > best_model_accuracy:
        best_model_accuracy = acc
        best_model_name = name
        best_model = model

print("-" * 40)
print(f"🏆 The Winner is '{best_model_name}' with {best_model_accuracy * 100:.2f}% accuracy!")

🏟️ Starting Model Competition...
----------------------------------------
🔹 Logistic Regression Accuracy: 95.60%
🔹 Support Vector Machine (SVM) Accuracy: 98.40%
🔹 Random Forest Accuracy: 97.80%
🔹 Gradient Boosting Accuracy: 97.00%
----------------------------------------
🏆 The Winner is 'Support Vector Machine (SVM)' with 98.40% accuracy!


In [5]:
# Cell 5: Detailed Report and Exporting the Final Model

print(f"📈 Detailed Classification Report for '{best_model_name}':")
best_predictions = best_model.predict(X_test_scaled)
print(classification_report(y_test, best_predictions, target_names=['NORMAL', 'WARNING', 'CRITICAL']))

# Create the 'src' directory if it doesn't exist
os.makedirs('src', exist_ok=True)

# File paths
model_path = 'src/best_crowd_model.pkl'
scaler_path = 'src/scaler.pkl'

# Export the winning model and scaler
joblib.dump(best_model, model_path)
joblib.dump(scaler, scaler_path)

print(f"✅ Step 5: Final model '{best_model_name}' has been successfully saved to '{model_path}'!")

📈 Detailed Classification Report for 'Support Vector Machine (SVM)':
              precision    recall  f1-score   support

      NORMAL       0.99      0.98      0.99       394
     WARNING       0.97      0.99      0.98       387
    CRITICAL       0.99      0.99      0.99       219

    accuracy                           0.98      1000
   macro avg       0.99      0.98      0.98      1000
weighted avg       0.98      0.98      0.98      1000

✅ Step 5: Final model 'Support Vector Machine (SVM)' has been successfully saved to 'src/best_crowd_model.pkl'!
